# 01 · Prompt engineering

This notebook covers:

1. Prompt templates whose variables are checked before rendering
2. Zero-shot versus few-shot message construction
3. Structured (JSON) output, parsed and validated
4. Comparing prompt versions against labelled examples

By default the notebook uses an offline mock model. Set `LLM_PROVIDER=openai` and
`OPENAI_API_KEY` to run the same code against a real model. The mock's answers do
not depend on wording, so the comparison in section 4 only shows the workflow.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from llm_utils import get_llm
from prompting import PromptTemplate, few_shot_messages, parse_json_output, validate_keys

llm = get_llm()
print("Provider:", llm.name)

## 1. Prompt templates with checked variables

In [ ]:
template = PromptTemplate(
    "You are a {role}. Answer in at most {max_sentences} sentences.\n"
    "Question: {question}"
)
print("Variables:", template.variables)
print(template.render(role="support agent", max_sentences=2,
                      question="How do I reset my password?"))

try:
    template.render(role="support agent", question="How do I reset my password?")
except KeyError as e:
    print("Caught:", e)

## 2. Zero-shot versus few-shot messages

In [ ]:
examples = [
    {"input": "Text: The delivery was fast and the box was intact.",
     "output": '{"sentiment": "positive"}'},
    {"input": "Text: The app crashes every time I open it.",
     "output": '{"sentiment": "negative"}'},
]
system = ('Classify the sentiment of the text. '
          'Respond only with JSON: {"sentiment": "positive" | "negative" | "neutral"}')

messages = few_shot_messages(system, examples, "Text: Support never answered my email.")
for m in messages:
    print(f"{m['role']:>9}: {m['content']}")

## 3. Structured output: parse and validate

In [ ]:
query = "Text: The new update is great and very reliable."
response = llm.chat(few_shot_messages(system, examples, query))
print("Raw reply:", response.content)

obj, err = parse_json_output(response.content)
print("Parsed:", obj, "| error:", err)
if obj is not None:
    print("Missing keys:", validate_keys(obj, ["sentiment"]))

# Failure handling: prose instead of JSON should be caught, not silently accepted.
obj, err = parse_json_output("Sure! The sentiment is positive.")
print("Bad output ->", obj, "| error:", err)

## 4. Comparing prompt versions on labelled examples

Change a prompt, measure on a fixed labelled set, and keep the version that scores
best without breaking the cases you already handle.

In [ ]:
labelled = [
    ("I love this phone, the battery is excellent.", "positive"),
    ("Terrible customer service, very slow replies.", "negative"),
    ("The package arrived on Tuesday.", "neutral"),
    ("Helpful staff and fast delivery, great experience.", "positive"),
]

prompt_versions = {
    "v1_terse": 'Classify sentiment as JSON: {"sentiment": ...}',
    "v2_detailed": ('You classify customer sentiment. Choose exactly one label: positive, '
                    'negative, or neutral. Respond only with JSON: {"sentiment": "<label>"}'),
}

def evaluate_prompt(system_prompt):
    correct = 0
    for text, gold in labelled:
        reply = llm.chat([{"role": "system", "content": system_prompt},
                          {"role": "user", "content": f"Text: {text}"}])
        obj, _ = parse_json_output(reply.content or "")
        if obj and obj.get("sentiment") == gold:
            correct += 1
    return correct / len(labelled)

for name, system_prompt in prompt_versions.items():
    print(f"{name}: accuracy = {evaluate_prompt(system_prompt):.2f}")

## Practices checklist

- State the role and the task in plain language.
- Specify the output format exactly, and validate it in code.
- Put constraints (length, allowed labels, what to avoid) in the prompt.
- Separate user content with clear delimiters so it cannot be confused with instructions.
- Add few-shot examples for the edge cases you care about most.
- Version prompts and measure each change on a labelled set (notebook 05).
- Never put secrets in prompts. Assume the prompt can be extracted by a user.